# P01 — El perceptrón: un modelo probabilístico de almacenamiento y organización de información en el cerebro

## 1. Título y paper

**Paper:** *The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain*  
**Autoría:** Frank Rosenblatt  
**Año y venue:** 1958 · Psychological Review, 65(6), 386–408  
**Nivel:** L1 · **Motor:** `perceptron`  
**Ficha completa:** [`P01_perceptron`](../../papers/foundational/P01_perceptron/README.md)

**Hito:** Primera máquina que aprende sus propios pesos a partir de ejemplos en lugar de ejecutar reglas escritas por una persona.

- [DOI (Psychological Review)](https://doi.org/10.1037/h0042519)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: La IA de los años 50 programaba reglas a mano; no existía un procedimiento para que un sistema ajustara su comportamiento observando datos.
2. Ejecutar una implementación mínima de la propuesta: Una unidad de decisión lineal con umbral y una regla de corrección de error que solo actúa cuando la predicción falla.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- McCulloch y Pitts (1943), neurona lógica de umbral
- Hebb (1949), plasticidad sináptica


## 4. Intuición

Una recta que separa dos grupos de puntos. El aprendizaje consiste en empujar la recta cada vez que un punto queda del lado equivocado. Nada más. Si los grupos se pueden separar con una recta, el empujón termina; si no, el empujón nunca termina.


## 5. Concepto mínimo

`ŷ = 1 si w·x + b ≥ 0, si no 0`. Regla de corrección: `w ← w + η(y − ŷ)x`, `b ← b + η(y − ŷ)`.

Solo se corrige ante error: si acierta, no toca nada. El teorema de Novikoff (1962) acota el número de correcciones por `(R/γ)²` cuando existe un margen `γ > 0`.


## 6. Código explicado

Veinte líneas bastan para el algoritmo de 1958. Lo que importa es la línea del `if`: sin error, no hay aprendizaje.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
def perceptron(data, epochs=20, lr=1.0):
    w, b, historial = [0.0, 0.0], 0.0, []
    for epoca in range(1, epochs + 1):
        errores = 0
        for x, y in data:
            z = w[0] * x[0] + w[1] * x[1] + b
            pred = 1 if z >= 0 else 0
            if pred != y:                      # <-- solo se aprende del error
                w = [wi + lr * (y - pred) * xi for wi, xi in zip(w, x)]
                b += lr * (y - pred)
                errores += 1
        historial.append({'epoca': epoca, 'errores': errores, 'w': list(w), 'b': b})
        if errores == 0:
            return {'converge': True, 'epocas': epoca, 'w': w, 'b': b, 'historial': historial}
    return {'converge': False, 'epocas': epochs, 'w': w, 'b': b, 'historial': historial[-3:]}

AND = [([0, 0], 0), ([0, 1], 0), ([1, 0], 0), ([1, 1], 1)]
XOR = [([0, 0], 0), ([0, 1], 1), ([1, 0], 1), ([1, 1], 0)]
show(perceptron(AND))

## 7. Predicción antes de ejecutar

Antes de ejecutar la celda siguiente, escribe tu respuesta:

1. ¿En cuántas épocas converge AND?
2. ¿Qué crees que hará XOR: converger en más épocas, o no converger nunca?
3. Si XOR no converge, ¿los pesos se quedan quietos o siguen cambiando?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
resultado_and = perceptron(AND)
resultado_xor = perceptron(XOR)
print('AND converge:', resultado_and['converge'], '· épocas:', resultado_and['epocas'])
print('XOR converge:', resultado_xor['converge'], '· épocas:', resultado_xor['epocas'])
print('XOR, últimas 3 épocas:')
show(resultado_xor['historial'])

## 9. Salida interpretable

AND converge y deja de moverse: `errores = 0`. XOR nunca llega a `errores = 0` y sus pesos siguen oscilando en las últimas épocas. **La oscilación es el dato**: no es que el algoritmo aprenda lento, es que no existe solución que buscar.


## 10. Comentario pedagógico

Esto separa dos ideas que los principiantes mezclan: *no converger por falta de épocas* (problema de presupuesto) y *no converger porque la clase de hipótesis no contiene la solución* (problema de capacidad representacional). Aumentar `epochs` a un millón no cambia el segundo caso.


## 11. Error o anti-patrón deliberado

Anti-patrón clásico: concluir «el modelo aprende» mirando solo el último `w` sin comprobar si el error llegó a cero.


In [ ]:
malo = perceptron(XOR, epochs=200)
print('pesos finales:', malo['w'], malo['b'])
print('conclusión apresurada: «ya está entrenado, tengo pesos»')

## 12. Corrección

La corrección es reportar siempre el criterio de parada junto con los pesos.


In [ ]:
def reportar(resultado, nombre):
    estado = 'CONVERGIÓ' if resultado['converge'] else 'NO CONVERGIÓ (tope de épocas)'
    print(f"{nombre}: {estado} · w={resultado['w']} b={resultado['b']}")

reportar(perceptron(AND), 'AND')
reportar(perceptron(XOR, epochs=200), 'XOR')

## 13. Desafío guiado

Añade la característica `x₃ = x₁·x₂` a XOR y vuelve a entrenar. ¿Se vuelve separable? Este es exactamente el truco que las capas ocultas aprenderán solas en P02.


In [ ]:
XOR3 = [([x[0], x[1], x[0] * x[1]], y) for x, y in XOR]

def perceptron3(data, epochs=20, lr=1.0):
    w, b = [0.0, 0.0, 0.0], 0.0
    for epoca in range(1, epochs + 1):
        errores = 0
        for x, y in data:
            pred = 1 if sum(wi * xi for wi, xi in zip(w, x)) + b >= 0 else 0
            if pred != y:
                w = [wi + lr * (y - pred) * xi for wi, xi in zip(w, x)]
                b += lr * (y - pred)
                errores += 1
        if errores == 0:
            return {'converge': True, 'epocas': epoca, 'w': w, 'b': b}
    return {'converge': False, 'epocas': epochs, 'w': w, 'b': b}

show(perceptron3(XOR3))

## 14. Desafío autónomo

Genera dos nubes de puntos gaussianas con distintos grados de solapamiento y mide cuántas correcciones necesita el perceptrón en función del margen. Contrasta tu curva empírica con la cota `(R/γ)²`. Documenta la semilla y el criterio de parada.


## 15. Evidencia de aprendizaje

Guarda: (a) el número de épocas de AND, (b) la evidencia de no convergencia de XOR, (c) el resultado de XOR con la característica `x₁·x₂`, y (d) una frase tuya distinguiendo «no converge todavía» de «no puede converger».

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P01_perceptron/README.md) · evaluación formal: [`assessments/papers/P01_perceptron.md`](../../assessments/papers/P01_perceptron.md)


## 16. Cierre

El perceptrón demostró que una máquina puede ajustar su propio comportamiento a partir de ejemplos. También demostró, sin quererlo, dónde estaba el techo: la frontera es lineal.


## 17. Conexión con el siguiente hito

- P02

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
